# How to Read Machine Learning Charts: A Detailed Guide

In this guide we will walk through how to correctly read and interpret
charts for the ROC-AUC, PR-AUC, and R-squared metrics with practical examples.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_curve, auc, precision_recall_curve, average_precision_score
import seaborn as sns

plt.style.use('seaborn-v0_8')
plt.rcParams['figure.figsize'] = (12, 8)

# 1. HOW TO READ THE ROC CURVE

In [ ]:
from sklearn.dummy import DummyClassifier

print("\n🎯 1. ROC CURVE (Receiver Operating Characteristic)")
print("-" * 50)

# Create data for the demonstration
X, y = make_classification(n_samples=1000, n_features=10, n_informative=5, 
                          n_redundant=2, n_clusters_per_class=1, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Train different models for comparison
good_model = RandomForestClassifier(n_estimators=100, random_state=42)
poor_model = DummyClassifier(strategy="stratified", random_state=42)  # Weak model, slightly better than random

good_model.fit(X_train, y_train)
poor_model.fit(X_train, y_train)

# Get predictions
good_proba = good_model.predict_proba(X_test)[:, 1]
poor_proba = poor_model.predict_proba(X_test)[:, 1]

# Compute ROC curves
good_fpr, good_tpr, _ = roc_curve(y_test, good_proba)
poor_fpr, poor_tpr, _ = roc_curve(y_test, poor_proba)

good_auc = auc(good_fpr, good_tpr)
poor_auc = auc(poor_fpr, poor_tpr)

## Chart 1: The main ROC curve with explanations

In [ ]:
plt.figure(figsize=(12, 8))
plt.plot(good_fpr, good_tpr, linewidth=3, label=f'Good model (AUC = {good_auc:.3f})', color='green')
plt.plot(poor_fpr, poor_tpr, linewidth=3, label=f'Weak model (AUC = {poor_auc:.3f})', color='red')
plt.plot([0, 1], [0, 1], 'k--', linewidth=2, label='Random prediction (AUC = 0.5)')

# Add annotations for explanation
plt.annotate('Ideal point\n(0, 1)', xy=(0, 1), xytext=(0.2, 0.8),
             arrowprops=dict(arrowstyle='->', color='blue', lw=2),
             fontsize=12, color='blue', weight='bold')

plt.annotate('Worst point\n(1, 0)', xy=(1, 0), xytext=(0.7, 0.3),
             arrowprops=dict(arrowstyle='->', color='red', lw=2),
             fontsize=12, color='red', weight='bold')

plt.fill_between(good_fpr, good_tpr, alpha=0.3, color='green', label='Area under the curve (AUC)')

plt.xlabel('False Positive Rate (FPR)\n= Fraction of false positives\n= 1 - Specificity', fontsize=12)
plt.ylabel('True Positive Rate (TPR)\n= Fraction of true positives\n= Sensitivity', fontsize=12)
plt.title('ROC Curve: How to read it', fontsize=14, weight='bold')
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)
plt.show()

print("📖 HOW TO READ THE ROC CURVE:")
print("• X-axis (FPR): Fraction of negative examples classified incorrectly")
print("• Y-axis (TPR): Fraction of positive examples classified correctly")
print("• Diagonal line: Random prediction")
print("• The further left and up the curve is, the better the model")
print("• AUC = area under the curve (from 0 to 1)")

## Chart 2: What different ROC curve shapes mean

In [ ]:
plt.figure(figsize=(12, 8))
plt.plot([0, 0, 1], [0, 1, 1], 'g-', linewidth=3, label='Ideal model (AUC = 1.0)')
plt.plot([0, 1], [0, 1], 'k--', linewidth=2, label='Random model (AUC = 0.5)')
plt.plot([0, 1, 1], [0, 0, 1], 'r-', linewidth=3, label='Worst model (AUC = 0.0)')

# Add examples of realistic curves
realistic_fpr = np.array([0, 0.1, 0.3, 0.6, 1.0])
realistic_tpr = np.array([0, 0.6, 0.8, 0.9, 1.0])
plt.plot(realistic_fpr, realistic_tpr, 'b-', linewidth=3, label='Realistic good model')

plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('Typical ROC curve shapes', fontsize=14, weight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("\n🎯 INTERPRETING ROC CURVE SHAPES:")
print("• Rectangular (ideal): First grows vertically, then horizontally")
print("• Bowed upward: Good model, quickly finds positive examples")
print("• Diagonal: Random prediction, the model does not work")
print("• Bowed downward: Bad model, but predictions can be inverted")

# 2. HOW TO READ THE PR CURVE

In [ ]:
# Create an imbalanced dataset
X_imb, y_imb = make_classification(n_samples=1000, n_features=10, n_informative=5,
                                  weights=[0.9, 0.1], random_state=42)
X_train_imb, X_test_imb, y_train_imb, y_test_imb = train_test_split(
    X_imb, y_imb, test_size=0.3, random_state=42)

# Train the models
good_model_imb = RandomForestClassifier(n_estimators=100, random_state=42)
poor_model_imb = LogisticRegression(C=0.01, random_state=42)

good_model_imb.fit(X_train_imb, y_train_imb)
poor_model_imb.fit(X_train_imb, y_train_imb)

good_proba_imb = good_model_imb.predict_proba(X_test_imb)[:, 1]
poor_proba_imb = poor_model_imb.predict_proba(X_test_imb)[:, 1]

# Compute PR curves
good_precision, good_recall, _ = precision_recall_curve(y_test_imb, good_proba_imb)
poor_precision, poor_recall, _ = precision_recall_curve(y_test_imb, poor_proba_imb)

good_pr_auc = average_precision_score(y_test_imb, good_proba_imb)
poor_pr_auc = average_precision_score(y_test_imb, poor_proba_imb)

baseline = y_test_imb.mean()

## Chart 3: PR curve with explanations

In [ ]:
plt.figure(figsize=(12, 8))
plt.plot(good_recall, good_precision, linewidth=3, 
         label=f'Good model (PR-AUC = {good_pr_auc:.3f})', color='green')
plt.plot(poor_recall, poor_precision, linewidth=3,
         label=f'Weak model (PR-AUC = {poor_pr_auc:.3f})', color='red')
plt.axhline(y=baseline, color='k', linestyle='--', linewidth=2,
            label=f'Baseline (fraction of positives = {baseline:.3f})')

# Fill the area under the curve
plt.fill_between(good_recall, good_precision, alpha=0.3, color='green')

# Add annotations
plt.annotate('Ideal point\n(1, 1)', xy=(1, 1), xytext=(0.7, 0.8),
             arrowprops=dict(arrowstyle='->', color='blue', lw=2),
             fontsize=12, color='blue', weight='bold')

plt.xlabel('Recall', fontsize=12)
plt.ylabel('Precision', fontsize=12)
plt.title('PR Curve: How to read it', fontsize=14, weight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("📖 HOW TO READ THE PR CURVE:")
print("• X-axis (Recall): What fraction of all positive examples we found")
print("• Y-axis (Precision): What fraction of our positive predictions is correct")
print("• Baseline: The fraction of positive examples in the data. This is the minimum bar the model must beat!")
print("• The further right and up the curve is, the better the model")
print("• Especially important for imbalanced data")

# 3. HOW TO READ REGRESSION CHARTS

In [ ]:
# Create data for regression
np.random.seed(42)
n_samples = 300
X_reg = np.random.randn(n_samples, 1)
y_perfect = 2 * X_reg.ravel() + 1  # Perfect linear relationship
y_good = y_perfect + np.random.randn(n_samples) * 0.5  # With a small amount of noise
y_poor = y_perfect + np.random.randn(n_samples) * 2    # With a large amount of noise

## Chart 4: Predicted vs Actual for different R² values

In [ ]:
plt.figure(figsize=(12, 8))
plt.scatter(y_perfect, y_perfect, alpha=0.7, label='Ideal model (R² = 1.0)', color='green')
plt.scatter(y_good, y_perfect, alpha=0.7, label='Good model (R² ≈ 0.8)', color='blue')
plt.scatter(y_poor, y_perfect, alpha=0.7, label='Weak model (R² ≈ 0.2)', color='red')

# Add the ideal line
min_val = min(y_poor.min(), y_perfect.min())
max_val = max(y_poor.max(), y_perfect.max())
plt.plot([min_val, max_val], [min_val, max_val], 'k--', linewidth=2, label='Ideal line')

plt.xlabel('Predicted values', fontsize=12)
plt.ylabel('Actual values', fontsize=12)
plt.title('Predicted vs Actual: How to read it', fontsize=14, weight='bold')
plt.legend()
plt.grid(True, alpha=0.3)

# Add annotations
plt.annotate('Ideal predictions\nlie on this line', 
             xy=(0, 0), xytext=(2, -2),
             arrowprops=dict(arrowstyle='->', color='black', lw=2),
             fontsize=10, weight='bold')
plt.show()

print("📖 HOW TO READ PREDICTED VS ACTUAL:")
print("• Each point = one observation")
print("• X-axis: What the model predicted")
print("• Y-axis: The actual value")
print("• Diagonal line: Ideal predictions")
print("• The closer the points are to the diagonal, the better the model")
print("• The scatter of the points shows the model's errors")

## Chart 5: Residual analysis

In [ ]:
residuals_good = y_perfect - y_good
residuals_poor = y_perfect - y_poor

plt.figure(figsize=(12, 8))
plt.scatter(y_good, residuals_good, alpha=0.7, color='blue', label='Good model')
plt.scatter(y_poor, residuals_poor, alpha=0.7, color='red', label='Weak model')
plt.axhline(y=0, color='black', linestyle='-', linewidth=2, label='Zero line')

plt.xlabel('Predicted values', fontsize=12)
plt.ylabel('Residuals (Actual - Predicted)', fontsize=12)
plt.title('Residual analysis: How to read it', fontsize=14, weight='bold')
plt.legend()
plt.grid(True, alpha=0.3)

# Add annotations for the residuals
plt.annotate('Ideal residuals = 0\n(model is accurate)', 
             xy=(0, 0), xytext=(2, 3),
             arrowprops=dict(arrowstyle='->', color='green', lw=2),
             fontsize=10, color='green', weight='bold')
plt.show()

print("\n📖 HOW TO READ THE RESIDUAL CHART:")
print("• Residual = Actual value - Predicted value")
print("• X-axis: Predicted values")
print("• Y-axis: Residuals (errors)")
print("• Zero line: Ideal predictions")
print("• Random scatter around zero = good model")
print("• Patterns in the residuals = problems with the model")

## Chart 6: Demonstrating different R² values

In [ ]:
x_demo = np.linspace(0, 10, 100)
y_r2_high = 2*x_demo + 1 + np.random.randn(100) * 0.5  # R² ≈ 0.9
y_r2_med = 2*x_demo + 1 + np.random.randn(100) * 2     # R² ≈ 0.5
y_r2_low = 2*x_demo + 1 + np.random.randn(100) * 5     # R² ≈ 0.1

plt.figure(figsize=(12, 8))
plt.scatter(x_demo, y_r2_high, alpha=0.6, label='High R² (≈0.9)', color='green')
plt.scatter(x_demo, y_r2_med, alpha=0.6, label='Medium R² (≈0.5)', color='orange')
plt.scatter(x_demo, y_r2_low, alpha=0.6, label='Low R² (≈0.1)', color='red')
plt.plot(x_demo, 2*x_demo + 1, 'k-', linewidth=2, label='True relationship')

plt.xlabel('Independent variable (X)', fontsize=12)
plt.ylabel('Dependent variable (Y)', fontsize=12)
plt.title('Different R² values: Visual comparison', fontsize=14, weight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("\n📖 HOW TO VISUALLY ASSESS R²:")
print("• R² close to 1: Points cluster tightly around the trend line")
print("• R² around 0.5: Noticeable scatter, but the trend is visible")
print("• R² close to 0: Strong scatter, the relationship is barely visible")
